# DataCotas - Sistema Completo de Análise

Este notebook demonstra as funcionalidades do sistema DataCotas:
1. **Análise de Cor de Pele**: Determinação de scores fenotípicos usando escala Monk
2. **Sistema de Cotas**: API REST para gerenciamento de inscrições cotistas
3. **Integração**: Análise combinada de dados do sistema

## 1. Importações e Configurações

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from skimage import color
import math
import os
import glob
import csv
from pathlib import Path

# Configurações de visualização
plt.style.use('dark_background')
%matplotlib inline

## 2. Escala Monk Contínua (91 Tons)

In [ ]:
# Escala Monk original (10 tons âncora)
MONK_SCALE_HEX_ORIGINAL = [
    "#f6ede4", "#f3e7db", "#f7ead0", "#eadaba", "#d7bd96", 
    "#a07e56", "#825c43", "#604134", "#3a312a", "#292420"
]

def hex_to_rgb(hex_color):
    """Converte cor hexadecimal para RGB."""
    hex_color = hex_color.lstrip('#')
    return np.array([int(hex_color[i:i+2], 16) for i in (0, 2, 4)])

def rgb_to_hex(rgb):
    """Converte cor RGB para hexadecimal."""
    r, g, b = np.clip(rgb, 0, 255)
    return '#{:02x}{:02x}{:02x}'.format(int(r), int(g), int(b))

# Gera tons intermediários por interpolação linear
EXTENDED_MONK_HEX = []
EXTENDED_MONK_TEXT = []
EXTENDED_MONK_RGB = []

for i in range(len(MONK_SCALE_HEX_ORIGINAL) - 1):
    c1 = hex_to_rgb(MONK_SCALE_HEX_ORIGINAL[i])
    c2 = hex_to_rgb(MONK_SCALE_HEX_ORIGINAL[i+1])
    
    for step in range(10):
        fraction = step / 10.0
        c_inter = c1 * (1.0 - fraction) + c2 * fraction
        
        EXTENDED_MONK_RGB.append(c_inter)
        EXTENDED_MONK_HEX.append(rgb_to_hex(c_inter))
        
        score_value = (i + 1) + fraction
        EXTENDED_MONK_TEXT.append(f"MST {score_value:.1f}")

# Inclui o último tom âncora
ultimo_tom = hex_to_rgb(MONK_SCALE_HEX_ORIGINAL[-1])
EXTENDED_MONK_RGB.append(ultimo_tom)
EXTENDED_MONK_HEX.append(rgb_to_hex(ultimo_tom))
EXTENDED_MONK_TEXT.append("MST 10.0")

# Pré-computa em CIELAB (espaço mais adequado para distância perceptual)
EXTENDED_MONK_LAB = [color.rgb2lab(np.uint8([[rgb]]))[0][0] for rgb in EXTENDED_MONK_RGB]

print(f"Escala Monk gerada com {len(EXTENDED_MONK_HEX)} tons")
print(f"Intervalo: MST 1.0 a MST 10.0")

## 3. Visualização da Escala Monk

In [ ]:
# Visualiza a escala Monk completa
fig, ax = plt.subplots(figsize=(15, 3))
scale_image = np.zeros((100, len(EXTENDED_MONK_HEX), 3), dtype=np.uint8)

for i, hex_color in enumerate(EXTENDED_MONK_HEX):
    rgb = hex_to_rgb(hex_color)
    scale_image[:, i:i+1] = rgb

ax.imshow(scale_image)
ax.set_title('Escala Monk Contínua (91 Tons)', fontsize=14, fontweight='bold')
ax.set_xlabel('Score Fenotípico (MST)')
ax.set_yticks([])
ax.set_xticks([0, len(EXTENDED_MONK_HEX)//4, len(EXTENDED_MONK_HEX)//2, 
              3*len(EXTENDED_MONK_HEX)//4, len(EXTENDED_MONK_HEX)-1])
ax.set_xticklabels(['MST 1.0', 'MST 3.25', 'MST 5.5', 'MST 7.75', 'MST 10.0'])
plt.tight_layout()
plt.show()

## 4. Funções de Processamento de Imagem

In [ ]:
def get_skin_color_smart(image, k=5):
    """Extrai um tom de pele base via K-Means.
    
    Seleciona o 2º cluster mais claro (em L* do LAB) para reduzir impacto de
    highlight estourado e de sombras/barba.
    """
    img_small = cv2.resize(image, (64, 64))
    pixels = img_small.reshape((-1, 3))
    
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(pixels)
    colors_rgb = kmeans.cluster_centers_.astype(int)
    
    colors_lab = [color.rgb2lab(np.uint8([[rgb]]))[0][0] for rgb in colors_rgb]
    
    luminance_rgb_pairs = []
    for i in range(k):
        lum = colors_lab[i][0]
        rgb = colors_rgb[i]
        luminance_rgb_pairs.append((lum, rgb))
    
    # 2º mais claro: equilíbrio entre superexposição e regiões escuras
    luminance_rgb_pairs.sort(key=lambda x: x[0], reverse=True)
    return luminance_rgb_pairs[1][1]

def calculate_lab_distance(color1_lab, color2_lab):
    """Distância euclidiana em CIELAB."""
    return math.sqrt(sum((c1 - c2) ** 2 for c1, c2 in zip(color1_lab, color2_lab)))

def match_monk_scale(dominant_rgb):
    """Encontra o melhor match na escala Monk."""
    dominant_lab = color.rgb2lab(np.uint8([[dominant_rgb]]))[0][0]
    
    distances = []
    for i, monk_lab in enumerate(EXTENDED_MONK_LAB):
        dist = calculate_lab_distance(dominant_lab, monk_lab)
        distances.append((dist, i))
    
    distances.sort()
    best_match_idx = distances[0][1]
    
    return EXTENDED_MONK_TEXT[best_match_idx], EXTENDED_MONK_HEX[best_match_idx], dominant_lab

## 5. Pipeline de Análise de Imagem

In [ ]:
def process_image(image_path, visualize=True):
    """Processa uma imagem e extrai a cor de pele."""
    img = cv2.imread(image_path)
    if img is None:
        print(f"Erro ao carregar imagem: {os.path.basename(image_path)}")
        return None
        
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    
    # Detecta rosto
    cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    face_cascade = cv2.CascadeClassifier(cascade_path)
    
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, 
                                          minNeighbors=6, 
                                          minSize=(100, 100))
    if len(faces) == 0:
        print(f"Nenhum rosto detectado: {os.path.basename(image_path)}")
        return None
        
    largest_face = max(faces, key=lambda rect: rect[2] * rect[3])
    x, y, box_w, box_h = largest_face
    
    # Define ROI nas bochechas
    cheek_l_x = x + int(box_w * 0.15)
    cheek_l_y = y + int(box_h * 0.45)
    cheek_w = int(box_w * 0.25)
    cheek_h = int(box_h * 0.25)
    
    cheek_r_x = x + int(box_w * 0.60)
    cheek_r_y = cheek_l_y 
    
    roi_left = img_rgb[cheek_l_y:cheek_l_y+cheek_h, cheek_l_x:cheek_l_x+cheek_w]
    roi_right = img_rgb[cheek_r_y:cheek_r_y+cheek_h, cheek_r_x:cheek_r_x+cheek_w]
    
    if roi_left.size == 0 or roi_right.size == 0:
        return None

    face_roi_combined = np.hstack((roi_left, roi_right))

    dominant_rgb = get_skin_color_smart(face_roi_combined)
    hex_dominant = '#%02x%02x%02x' % tuple(dominant_rgb)
    
    # Matching na escala MST
    monk_text, monk_hex, dominant_lab = match_monk_scale(dominant_rgb)
    
    if visualize:
        # Visualização
        fig = plt.figure(figsize=(12, 7))
        fig.suptitle(f'DataCotas - Análise: {os.path.basename(image_path)}', 
                    fontsize=16, fontweight='bold')
        
        ax1 = fig.add_subplot(1, 4, 1)
        img_plot = img_rgb.copy()
        cv2.rectangle(img_plot, (cheek_l_x, cheek_l_y), 
                     (cheek_l_x+cheek_w, cheek_l_y+cheek_h), (0, 255, 0), 3)
        cv2.rectangle(img_plot, (cheek_r_x, cheek_r_y), 
                     (cheek_r_x+cheek_w, cheek_r_y+cheek_h), (0, 255, 0), 3)
        ax1.imshow(img_plot)
        ax1.axis('off')
        ax1.set_title("Captura (Bochechas)")
        
        ax2 = fig.add_subplot(1, 4, 2)
        ax2.imshow(face_roi_combined)
        ax2.axis('off')
        ax2.set_title("Área Combinada")
        
        ax3 = fig.add_subplot(1, 4, 3)
        dom_color_img = np.zeros((100, 100, 3), dtype=np.uint8)
        dom_color_img[:] = dominant_rgb
        ax3.imshow(dom_color_img)
        ax3.axis('off')
        ax3.set_title(f"Cor Extraída\n{hex_dominant}")
        
        ax4 = fig.add_subplot(1, 4, 4)
        monk_color_img = np.zeros((100, 100, 3), dtype=np.uint8)
        monk_color_img[:] = [int(monk_hex.lstrip('#')[i:i+2], 16) 
                           for i in (0, 2, 4)]
        ax4.imshow(monk_color_img)
        ax4.axis('off')
        ax4.set_title(f"Score Phenotípico:\n{monk_text}\n({monk_hex})")
        
        lab_text = f"Valores LAB: L*={dominant_lab[0]:.1f}, " \
                  f"a*={dominant_lab[1]:.1f}, b*={dominant_lab[2]:.1f}"
        plt.figtext(0.5, 0.05, lab_text, ha="center", fontsize=11, 
                   bbox={"facecolor":"#444", "alpha":0.7, "pad":5})
        
        plt.tight_layout(rect=[0, 0.1, 1, 0.95])
        plt.show()

    return {
        'Candidato': os.path.basename(image_path),
        'Cor_Extraida_HEX': hex_dominant,
        'Monk_Score': monk_text,
        'Monk_HEX': monk_hex,
        'LAB_L': round(dominant_lab[0], 2),
        'LAB_a': round(dominant_lab[1], 2),
        'LAB_b': round(dominant_lab[2], 2)
    }

## 6. Processamento em Lote

In [ ]:
def process_batch(folder_path, output_csv='relatorio_banca.csv', visualize=True):
    """Processa todas as imagens em uma pasta e gera relatório CSV."""
    if not os.path.exists(folder_path):
        print(f"Erro: A pasta '{folder_path}' não foi encontrada.")
        return None
    
    extensoes = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
    fotos = []
    for ext in extensoes:
        fotos.extend(glob.glob(os.path.join(folder_path, ext)))
    
    if not fotos:
        print(f"Nenhuma imagem encontrada na pasta '{folder_path}'.")
        return None
    
    print(f"Iniciando análise de {len(fotos)} candidatos...")
    resultados = []
    
    for i, caminho_da_foto in enumerate(fotos, 1):
        print(f"[{i}/{len(fotos)}] Processando: {os.path.basename(caminho_da_foto)}")
        dados = process_image(caminho_da_foto, visualize=visualize)
        if dados:
            resultados.append(dados)
    
    if resultados:
        colunas = ['Candidato', 'Cor_Extraida_HEX', 'Monk_Score', 
                   'Monk_HEX', 'LAB_L', 'LAB_a', 'LAB_b']
        try:
            with open(output_csv, mode='w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=colunas)
                writer.writeheader()
                writer.writerows(resultados)
            print(f"\nSucesso! Relatório salvo em: '{output_csv}'")
            return resultados
        except Exception as e:
            print(f"\nErro CSV: {e}")
            return None
    else:
        print("\nNenhum resultado válido obtido.")
        return None

## 7. Exemplo de Uso

In [ ]:
# Exemplo: Processar uma única imagem
# imagem_exemplo = "caminho/para/imagem.jpg"
# resultado = process_image(imagem_exemplo, visualize=True)
# print(resultado)

# Exemplo: Processar lote de imagens
# pasta_candidatos = "caminho/para/pasta/candidatos"
# resultados = process_batch(pasta_candidatos, output_csv='relatorio.csv', visualize=False)

print("Configure o caminho das imagens acima e execute as células.")

## 8. Análise de Resultados

In [ ]:
def analyze_results(results):
    """Gera estatísticas dos resultados."""
    if not results:
        print("Nenhum resultado para analisar.")
        return
    
    # Converte Monk Scores para valores numéricos
    monk_values = [float(r['Monk_Score'].split()[1]) for r in results]
    
    print("=== Estatísticas da Análise ===")
    print(f"Total de candidatos: {len(results)}")
    print(f"Média Monk Score: {np.mean(monk_values):.2f}")
    print(f"Mediana Monk Score: {np.median(monk_values):.2f}")
    print(f"Desvio Padrão: {np.std(monk_values):.2f}")
    print(f"Mínimo: {np.min(monk_values):.2f}")
    print(f"Máximo: {np.max(monk_values):.2f}")
    
    # Histograma
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(monk_values, bins=20, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Monk Score (MST)')
    ax.set_ylabel('Frequência')
    ax.set_title('Distribuição de Scores Fenotípicos')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Exemplo de uso:
# analyze_results(resultados)

## 10. Integração com API Django REST

### 10.1 Configuração da API

In [ ]:
import requests\nimport pandas as pd\nfrom datetime import datetime\nimport json\n\n# Configurações da API\nAPI_BASE_URL = \"http://localhost:8000/api\"  # Ajuste conforme necessário\n\nprint("=== Integração com API Django REST ===")\nprint(f"API Base URL: {API_BASE_URL}")

### 10.2 Funções de Integração com API

In [ ]:
def get_modalidades():\n    \"\"\"Busca todas as modalidades de cota.\"\"\"\n    try:\n        response = requests.get(f\"{API_BASE_URL}/modalidades/\")\n        response.raise_for_status()\n        return response.json()\n    except requests.exceptions.RequestException as e:\n        print(f\"Erro ao buscar modalidades: {e}\")\n        return []\n\ndef get_inscricoes():\n    \"\"\"Busca todas as inscrições.\"\"\"\n    try:\n        response = requests.get(f\"{API_BASE_URL}/inscricoes/\")\n        response.raise_for_status()\n        return response.json()\n    except requests.exceptions.RequestException as e:\n        print(f\"Erro ao buscar inscrições: {e}\")\n        return []\n\ndef criar_inscricao(dados_inscricao):\n    \"\"\"Cria uma nova inscrição.\"\"\"\n    try:\n        response = requests.post(\n            f\"{API_BASE_URL}/inscricoes/\",\n            json=dados_inscricao,\n            headers={'Content-Type': 'application/json'}\n        )\n        response.raise_for_status()\n        return response.json()\n    except requests.exceptions.RequestException as e:\n        print(f\"Erro ao criar inscrição: {e}\")\n        return None\n\nprint(\"Funções de API configuradas.\")

### 10.3 Buscar e Visualizar Modalidades

In [ ]:
# Buscar modalidades disponíveis\nmodalidades = get_modalidades()\n\nif modalidades:\n    df_modalidades = pd.DataFrame(modalidades)\n    print(\"=== Modalidades Disponíveis ===\")\n    print(df_modalidades.to_string(index=False))\n    \n    # Visualização\n    fig, ax = plt.subplots(figsize=(10, 4))\n    ax.axis('tight')\n    ax.axis('off')\n    table = ax.table(cellText=df_modalidades.values,\n                     colLabels=df_modalidades.columns,\n                     cellLoc='center',\n                     loc='center')\n    table.auto_set_font_size(False)\n    table.set_fontsize(10)\n    table.scale(1.2, 1.5)\n    ax.set_title('Modalidades de Cota Disponíveis', fontsize=14, fontweight='bold')\n    plt.tight_layout()\n    plt.show()\nelse:\n    print(\"Nenhuma modalidade encontrada. Verifique se a API está rodando.\")

### 10.4 Buscar e Analisar Inscrições

In [ ]:
# Buscar todas as inscrições\ninscricoes = get_inscricoes()\n\nif inscricoes:\n    df_inscricoes = pd.DataFrame(inscricoes)\n    print(f\"=== Total de Inscrições: {len(df_inscricoes)} ===\")\n    print(\"\\nColunas disponíveis:\", df_inscricoes.columns.tolist())\n    print(\"\\nPrimeiras 5 inscrições:\")\n    print(df_inscricoes[['id', 'nome', 'cpf', 'modalidade_nome', 'sexo']].head())\nelse:\n    print(\"Nenhuma inscrição encontrada. Verifique se a API está rodando.\")

In [ ]:
def analisar_inscricoes_por_modalidade(df_inscricoes):\n    \"\"\"Gera estatísticas por modalidade de cota.\"\"\"\n    if df_inscricoes.empty:\n        return None\n    \n    # Contagem por modalidade\n    contagem_modalidade = df_inscricoes['modalidade_nome'].value_counts()\n    \n    # Distribuição por sexo\n    distribuicao_sexo = df_inscricoes.groupby(['modalidade_nome', 'sexo']).size().unstack(fill_value=0)\n    \n    return {\n        'contagem_modalidade': contagem_modalidade,\n        'distribuicao_sexo': distribuicao_sexo\n    }\n\n# Análise das inscrições\nif 'df_inscricoes' in locals() and not df_inscricoes.empty:\n    analise = analisar_inscricoes_por_modalidade(df_inscricoes)\n    \n    if analise:\n        print(\"=== Inscrições por Modalidade ===\")\n        print(analise['contagem_modalidade'])\n        \n        print(\"\\n=== Distribuição por Sexo ===\")\n        print(analise['distribuicao_sexo'])

### 10.5 Visualização de Inscrições por Modalidade

In [ ]:
# Visualização das inscrições por modalidade\nif 'analise' in locals() and analise:\n    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))\n    \n    # Gráfico 1: Inscrições por modalidade\n    modalidades = analise['contagem_modalidade'].index.tolist()\n    contagens = analise['contagem_modalidade'].values.tolist()\n    \n    colors = plt.cm.Set3(range(len(modalidades)))\n    ax1.bar(modalidades, contagens, color=colors)\n    ax1.set_title('Inscrições por Modalidade de Cota', fontsize=12, fontweight='bold')\n    ax1.set_xlabel('Modalidade')\n    ax1.set_ylabel('Quantidade')\n    ax1.tick_params(axis='x', rotation=45)\n    ax1.grid(True, alpha=0.3)\n    \n    # Gráfico 2: Distribuição por sexo\n    analise['distribuicao_sexo'].plot(kind='bar', ax=ax2, stacked=True)\n    ax2.set_title('Distribuição por Sexo (por Modalidade)', fontsize=12, fontweight='bold')\n    ax2.set_xlabel('Modalidade')\n    ax2.set_ylabel('Quantidade')\n    ax2.tick_params(axis='x', rotation=45)\n    ax2.legend(title='Sexo', bbox_to_anchor=(1.05, 1), loc='upper left')\n    ax2.grid(True, alpha=0.3)\n    \n    plt.tight_layout()\n    plt.show()

### 10.6 Exemplo de Criação de Inscrição

In [ ]:
# Exemplo de criação de inscrição para Aluno PCD\nexemplo_inscricao_pcd = {\n    \"modalidade\": 2,  # ID da modalidade Aluno PCD\n    \"nome\": \"João Silva\",\n    \"rg\": \"123456789\",\n    \"cpf\": \"12345678901\",\n    \"sexo\": \"M\",\n    \"dados_pcd\": {\n        \"codigo_cid\": \"G80.9\",\n        \"laudo_medico\": None  # Upload de arquivo seria feito via multipart/form-data\n    }\n}\n\n# Exemplo de criação de inscrição para Filhos de Agentes de Segurança\nexemplo_inscricao_agentes = {\n    \"modalidade\": 1,  # ID da modalidade Filhos de Agentes de Segurança\n    \"nome\": \"Maria Santos\",\n    \"rg\": \"987654321\",\n    \"cpf\": \"98765432109\",\n    \"sexo\": \"F\",\n    \"dados_filhos_agentes\": {\n        \"cad_unico\": None,\n        \"decisao_administrativa\": None,\n        \"certidao_obito\": None,\n        \"comprovante_reforma_pensao\": None\n    }\n}\n\nprint(\"Exemplos de inscrições configurados.\")\nprint(\"Para criar uma inscrição, use: criar_inscricao(exemplo_inscricao_pcd)\")\nprint(\"\\nNota: Upload de arquivos deve ser feito via multipart/form-data.\")

## 11. Integração: Análise de Cor de Pele + Sistema de Cotas

### 11.1 Workflow Completo: Análise de Imagem + Registro no Sistema

In [ ]:
def processar_e_registrar(image_path, dados_inscricao):\n    \"\"\"Workflow completo: analisa imagem e registra no sistema.\"\"\"\n    # 1. Analisa a cor de pele\n    resultado_analise = process_image(image_path, visualize=True)\n    \n    if not resultado_analise:\n        print(\"Falha na análise da imagem.\")\n        return None\n    \n    # 2. Adiciona dados da análise à inscrição\n    dados_inscricao['monk_score'] = resultado_analise['Monk_Score']\n    dados_inscricao['monk_hex'] = resultado_analise['Monk_HEX']\n    dados_inscricao['lab_l'] = resultado_analise['LAB_L']\n    dados_inscricao['lab_a'] = resultado_analise['LAB_a']\n    dados_inscricao['lab_b'] = resultado_analise['LAB_b']\n    \n    # 3. Registra no sistema\n    inscricao = criar_inscricao(dados_inscricao)\n    \n    if inscricao:\n        print(\"\\n=== Workflow Completo ===\")\n        print(f\"Análise de cor: {resultado_analise['Monk_Score']}\")\n        print(f\"Inscrição criada: ID {inscricao.get('id')}\")\n        return {\n            'analise': resultado_analise,\n            'inscricao': inscricao\n        }\n    else:\n        print(\"Falha ao criar inscrição.\")\n        return None\n\nprint(\"Workflow completo configurado.\")

### 11.2 Análise Estatística: Cor de Pele por Modalidade

In [ ]:
def analisar_cor_por_modalidade(df_inscricoes):\n    \"\"\"Analisa distribuição de scores Monk por modalidade.\"\"\"\n    if df_inscricoes.empty or 'monk_score' not in df_inscricoes.columns:\n        print(\"Dados insuficientes para análise.\")\n        return None\n    \n    # Extrai valores numéricos do Monk Score\n    df_inscricoes['monk_valor'] = df_inscricoes['monk_score'].str.extract(r'MST ([\\d.]+)').astype(float)\n    \n    # Estatísticas por modalidade\n    stats_por_modalidade = df_inscricoes.groupby('modalidade_nome')['monk_valor'].agg([\n        ('count', 'count'),\n        ('mean', 'mean'),\n        ('median', 'median'),\n        ('std', 'std'),\n        ('min', 'min'),\n        ('max', 'max')\n    ])\n    \n    return stats_por_modalidade\n\n# Exemplo de uso\nif 'df_inscricoes' in locals() and not df_inscricoes.empty:\n    stats = analisar_cor_por_modalidade(df_inscricoes)\n    if stats is not None:\n        print(\"=== Estatísticas de Cor de Pele por Modalidade ===\")\n        print(stats)

### 11.3 Visualização: Distribuição de Cor de Pele por Modalidade

In [ ]:
# Visualização da distribuição de cor de pele por modalidade\nif 'stats' in locals() and stats is not None:\n    fig, ax = plt.subplots(figsize=(12, 6))\n    \n    modalidades = stats.index.tolist()\n    medias = stats['mean'].values\n    \n    colors = plt.cm.viridis(range(len(modalidades)))\n    bars = ax.bar(modalidades, medias, color=colors, alpha=0.8)\n    \n    ax.set_title('Média de Score Monk por Modalidade de Cota', \n                fontsize=14, fontweight='bold')\n    ax.set_xlabel('Modalidade')\n    ax.set_ylabel('Média Monk Score')\n    ax.tick_params(axis='x', rotation=45)\n    ax.grid(True, alpha=0.3, axis='y')\n    \n    # Adiciona rótulos de valor nas barras\n    for bar, media in zip(bars, medias):\n        height = bar.get_height()\n        ax.text(bar.get_x() + bar.get_width()/2., height,\n                f'{media:.2f}',\n                ha='center', va='bottom', fontsize=10)\n    \n    plt.tight_layout()\n    plt.show()

## 12. Docker e Deploy

### 12.1 Como Rodar o Sistema com Docker

In [ ]:
# Instruções Docker para o sistema DataCotas\nprint(\"=== Como Rodar o Sistema com Docker ===\")\nprint(\"\")\nprint(\"1. Iniciar o banco de dados:\")\nprint(\"   sudo docker compose up -d db\")\nprint(\"\")\nprint(\"2. Executar migrações:\")\nprint(\"   sudo docker compose run --rm web python manage.py migrate\")\nprint(\"\")\nprint(\"3. Iniciar o servidor web:\")\nprint(\"   sudo docker compose up web\")\nprint(\"\")\nprint(\"Ou com venv local:\")\nprint(\"   pip install -r requirements.txt\")\nprint(\"   python manage.py migrate\")\nprint(\"   python manage.py runserver\")

## 13. Front-end Integration (Next.js)

### 13.1 Estrutura do Front-end

In [ ]:
# Estrutura do front-end Next.js\nprint(\"=== Estrutura do Front-end Next.js ===\")\nprint(\"\")\nprint(\"Diretório: front-end/\")\nprint(\"  ├── src/\")\nprint(\"  │   └── app/\")\nprint(\"  │       ├── layout.tsx       # Layout principal\")\nprint(\"  │       ├── page.tsx         # Página inicial\")\nprint(\"  │       └── globals.css      # Estilos globais\")\nprint(\"  ├── package.json\")\nprint(\"  ├── next.config.ts\")\nprint(\"  └── tsconfig.json\")\nprint(\"\")\nprint(\"Para rodar o front-end:\")\nprint(\"  cd front-end\")\nprint(\"  npm install\")\nprint(\"  npm run dev\")

### 13.2 Exemplo de Integração Front-end ↔ Backend

In [ ]:
# Exemplo de chamada da API do front-end (JavaScript/TypeScript)\nfrontend_api_example = '''\n// Exemplo de chamada da API no Next.js\n\nasync function getModalidades() {\n  const response = await fetch('http://localhost:8000/api/modalidades/');\n  const data = await response.json();\n  return data;\n}\n\nasync function criarInscricao(dados) {\n  const formData = new FormData();\n  \n  // Adiciona campos básicos\n  Object.keys(dados).forEach(key => {\n    if (key !== 'dados_pcd' && key !== 'dados_filhos_agentes') {\n      formData.append(key, dados[key]);\n    }\n  });\n  \n  // Adiciona arquivos específicos da modalidade\n  if (dados.dados_pcd) {\n    Object.keys(dados.dados_pcd).forEach(key => {\n      if (dados.dados_pcd[key] instanceof File) {\n        formData.append(key, dados.dados_pcd[key]);\n      }\n    });\n  }\n  \n  const response = await fetch('http://localhost:8000/api/inscricoes/', {\n    method: 'POST',\n    body: formData\n  });\n  \n  return await response.json();\n}\n'''\n\nprint("=== Exemplo de Integração Front-end ↔ Backend ===")\nprint(frontend_api_example)

## 14. Análise Avançada

### 14.1 Correlação entre Cor de Pele e Modalidade

In [ ]:
def analisar_correlacao_cor_modalidade(df_inscricoes):\n    \"\"\"Analisa correlação entre cor de pele e modalidade de cota.\"\"\"\n    if df_inscricoes.empty or 'monk_score' not in df_inscricoes.columns:\n        return None\n    \n    # Extrai valores numéricos do Monk Score\n    df_inscricoes['monk_valor'] = df_inscricoes['monk_score'].str.extract(r'MST ([\\d.]+)').astype(float)\n    \n    # Cria tabela de contingência\n    # Divide Monk Score em faixas\n    bins = [0, 3, 5, 7, 10]\n    labels = ['Claro (1-3)', 'Médio-Claro (3-5)', 'Médio-Escuro (5-7)', 'Escuro (7-10)']\n    df_inscricoes['faixa_cor'] = pd.cut(df_inscricoes['monk_valor'], \n                                         bins=bins, \n                                         labels=labels,\n                                         include_lowest=True)\n    \n    # Tabela de contingência\n    tabela_contingencia = pd.crosstab(\n        df_inscricoes['modalidade_nome'], \n        df_inscricoes['faixa_cor']\n    )\n    \n    return tabela_contingencia\n\n# Exemplo de uso\nif 'df_inscricoes' in locals() and not df_inscricoes.empty:\n    correlacao = analisar_correlacao_cor_modalidade(df_inscricoes)\n    if correlacao is not None:\n        print(\"=== Correlação Cor de Pele × Modalidade ===\")\n        print(correlacao)

### 14.2 Visualização de Correlação Cor × Modalidade

In [ ]:
# Visualização da correlação cor × modalidade\nif 'correlacao' in locals() and correlacao is not None:\n    fig, ax = plt.subplots(figsize=(12, 6))\n    \n    # Heatmap\n    im = ax.imshow(correlacao.values, cmap='YlOrRd', aspect='auto')\n    \n    # Configurações do eixo\n    ax.set_xticks(range(len(correlacao.columns)))\n    ax.set_yticks(range(len(correlacao.index)))\n    ax.set_xticklabels(correlacao.columns, rotation=45, ha='right')\n    ax.set_yticklabels(correlacao.index)\n    \n    # Adiciona valores nas células\n    for i in range(len(correlacao.index)):\n        for j in range(len(correlacao.columns)):\n            valor = correlacao.values[i, j]\n            if valor > 0:\n                text = ax.text(j, i, str(int(valor)),\n                             ha=\"center\", va=\"center\", color=\"black\",\n                             fontsize=10, fontweight='bold')\n    \n    # Colorbar\n    cbar = plt.colorbar(im, ax=ax)\n    cbar.set_label('Quantidade', rotation=270, labelpad=20)\n    \n    ax.set_title('Distribuição de Cor de Pele por Modalidade de Cota',\n                fontsize=14, fontweight='bold', pad=20)\n    plt.tight_layout()\n    plt.show()

## 15. Estrutura Completa do Projeto

### 15.1 Arquitetura do Sistema

In [ ]:
# Estrutura completa do projeto DataCotas\nprint(\"=== Arquitetura do Sistema DataCotas ===\")\nprint(\"\")\nprint(\"DataCotas/\")\nprint(\"├── backend/                    # Django REST API\")\nprint(\"│   ├── config/               # Configurações Django\")\nprint(\"│   ├── cotas/                # App principal\")\nprint(\"│   │   ├── models.py         # Modelos: Modalidade, InscricaoCota, etc.\")\nprint(\"│   │   ├── serializers.py    # Serializers DRF\")\nprint(\"│   │   ├── views.py          # Views REST\")\nprint(\"│   │   └── urls.py           # Rotas da API\")\nprint(\"│   ├── Dockerfile\")\nprint(\"│   └── docker-compose.yml\")\nprint(\"├── front-end/                  # Next.js Application\")\nprint(\"│   ├── src/app/\")\nprint(\"│   │   ├── layout.tsx\")\nprint(\"│   │   ├── page.tsx\")\nprint(\"│   │   └── globals.css\")\nprint(\"│   └── package.json\")\nprint(\"├── modulos/\")\nprint(\"│   └── reconhecimento_facial/ # Análise de cor de pele\")\nprint(\"│       ├── datacotas_mvp.py\")\nprint(\"│       └── datacotas_mvp1.py\")\nprint(\"└── docs/\")\nprint(\"    └── DESIGN.md\")

## 16. Exemplos de Uso Completo

### 16.1 Workflow 1: Análise de Imagem Isolada

In [ ]:
# Workflow 1: Análise de imagem isolada\nprint(\"=== Workflow 1: Análise de Imagem Isolada ===\")\nprint(\"\")\nprint(\"1. Carregar imagem:\")\nprint(\"   imagem = 'caminho/para/foto.jpg'\")\nprint(\"\")\nprint(\"2. Processar imagem:\")\nprint(\"   resultado = process_image(imagem, visualize=True)\")\nprint(\"\")\nprint(\"3. Ver resultado:\")\nprint(\"   print(resultado)\")\nprint(\"\")\nprint(\"Resultado esperado:\")\nprint(\"   {\")\nprint(\"     'Candidato': 'foto.jpg',\")\nprint(\"     'Cor_Extraida_HEX': '#d7bd96',\")\nprint(\"     'Monk_Score': 'MST 5.2',\")\nprint(\"     'Monk_HEX': '#d7bd96',\")\nprint(\"     'LAB_L': 65.3, 'LAB_a': 12.1, 'LAB_b': 23.4\")\nprint(\"   }\")

### 16.2 Workflow 2: Registro Completo no Sistema

In [ ]:
# Workflow 2: Registro completo no sistema\nprint(\"=== Workflow 2: Registro Completo no Sistema ===\")\nprint(\"\")\nprint(\"1. Analisar imagem de um candidato:\")\nprint(\"   imagem = 'caminho/para/foto_candidato.jpg'\")\nprint(\"   resultado_analise = process_image(imagem, visualize=True)\")\nprint(\"\")\nprint(\"2. Preparar dados da inscrição:\")\nprint(\"   dados_inscricao = {\")\nprint(\"       'modalidade': 2,  # ID da modalidade Aluno PCD\")\nprint(\"       'nome': 'João Silva',\")\nprint(\"       'rg': '123456789',\")\nprint(\"       'cpf': '12345678901',\")\nprint(\"       'sexo': 'M',\")\nprint(\"       'dados_pcd': {\")\nprint(\"           'codigo_cid': 'G80.9',\")\nprint(\"           'laudo_medico': None  # Upload via multipart\nprint(\"       }\")\nprint(\"   }\")\nprint(\"\")\nprint(\"3. Registrar no sistema:\")\nprint(\"   inscricao = criar_inscricao(dados_inscricao)\")\nprint(\"\")\nprint(\"4. Ver resultado:\")\nprint(\"   print(inscricao)\")

### 16.3 Workflow 3: Análise em Lote com Relatório

## 9. Informações do Sistema

In [ ]:
import sys
print("=== Informações do Sistema ===")
print(f"Python: {sys.version}")
print(f"OpenCV: {cv2.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {plt.__version__}")
print(f"Scikit-learn: {cv2.__version__}")
print(f"Scikit-image: {color.__version__}")